In [1]:
!git clone https://github.com/HenriqueSchmitz/mario-the-explorer

Cloning into 'mario-the-explorer'...
remote: Enumerating objects: 237, done.
remote: Counting objects: 100% (237/237), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 237 (delta 105), reused 186 (delta 63), pack-reused 0 (from 0)
Receiving objects: 100% (237/237), 798.23 KiB | 4.75 MiB/s, done.
Resolving deltas: 100% (105/105), done.


In [2]:
!sh ./mario-the-explorer/setup.sh

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 MB 167.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 389.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 310.1 MB/s eta 0:00:00
Importing SuperMarioWorld-Snes-v0
Imported 1 games


In [3]:
!pip install -q stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 58.7 MB/s eta 0:00:00


In [4]:
from typing import Optional
from enum import Enum
from logging import Logger

import torch
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.logger import KVWriter, Logger as PpoLogger

from mario_the_explorer import (MultiAttemptSuperMarioWorldEmulator, RewardModel, ScreenOverlay, Tile, get_file_logger,
                                tile_absolute_id, TileEncoder, SuperMarioAction, SuperMarioCombo, SuperMarioDiscretizer,
                                prime_policy_for_combo, TileType)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
class Direction(Enum):
    LEFT = 0
    RIGHT = 1
    UP = 2
    DOWN = 3

In [7]:
from mario_the_explorer.environment import tiles
class TryThingsRewardModel(RewardModel):
    def __init__(self):
        self._blocks_seen = set()
        self._block_action_counts = {}

    def reset(self) -> None:
        self._blocks_seen = set()
        self._block_action_counts = {}

    def get_reward(self,
                   action: list[int],
                   observation: list[list[Tile]],
                   terminated: bool,
                   truncated: bool,
                   info: dict) -> float:
        reward = 0.0
        for row in observation:
            for tile in row:
                tile_id = tile_absolute_id(tile)
                if tile_id not in self._blocks_seen:
                    self._blocks_seen.add(tile_id)
                    reward += 10.0
        tiles_around_mario = self._get_tiles_around_mario(observation)
        for tile_and_direction in tiles_around_mario:
            if tile_and_direction not in self._block_action_counts:
                self._block_action_counts[tile_and_direction] = 0
            self._block_action_counts[tile_and_direction] += 1
            reward += 1.0 / (self._block_action_counts[tile_and_direction]**2)
        return reward

    def _get_tiles_around_mario(self, observation: list[list[Tile]]) -> set[tuple[Direction, int]]:
        mario_coordinates = self._find_mario_coordinates(observation)
        blocks_around_mario = set()
        if not mario_coordinates:
            return blocks_around_mario
        for mario_row, mario_col in mario_coordinates:
            if mario_row > 0:
                block_above_mario = observation[mario_row - 1][mario_col]
                if block_above_mario["type"] != TileType.MARIO and block_above_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.UP.name, tile_absolute_id(block_above_mario)))
            if mario_row < len(observation) - 1:
                block_below_mario = observation[mario_row + 1][mario_col]
                if block_below_mario["type"] != TileType.MARIO and block_below_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.DOWN.name, tile_absolute_id(block_below_mario)))
            if mario_col > 0:
                block_left_of_mario = observation[mario_row][mario_col - 1]
                if block_left_of_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.LEFT.name, tile_absolute_id(block_left_of_mario)))
            if mario_col < len(observation[0]) - 1:
                block_right_of_mario = observation[mario_row][mario_col + 1]
                if block_right_of_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.RIGHT.name, tile_absolute_id(block_right_of_mario)))
        return blocks_around_mario


    def _find_mario_coordinates(self, observation: list[list[Tile]]) -> list[tuple[int, int]]:
        mario_coordinates = []
        for row_idx, row in enumerate(observation):
            for col_idx, tile in enumerate(row):
                if tile["type"] == TileType.MARIO:
                    mario_coordinates.append((row_idx, col_idx))
        return mario_coordinates

In [8]:
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
import torch.nn as nn

class CustomMarioCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Box, features_dim: int = 64):
        super().__init__(observation_space, features_dim)
        n_input_channels = observation_space.shape[0]

        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 24, kernel_size=3, stride=2),
            nn.ReLU(),
            nn.Conv2d(24, 32, kernel_size=3, stride=2),
            nn.ReLU(),
            nn.Flatten(),
        )

        # Compute shape by doing one forward pass
        with torch.no_grad():
            sample_tensor = torch.as_tensor(observation_space.sample()[None]).float()
            n_flatten = self.cnn(sample_tensor).shape[1]

        self.linear = nn.Sequential(
            nn.Linear(n_flatten, features_dim),
            nn.ReLU()
        )

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        return self.linear(self.cnn(observations))

In [9]:
RUN_NAME = "cnn_action_rewards"
LEVEL = "DonutPlains1"
LOG_LEVEL = "INFO"
ATTEMPTS = 1

In [10]:
class PpoKvWriter(KVWriter):
    def __init__(self, logger: Logger):
        self._logger = logger

    def write(self, key_values, key_excluded, step=0):
        for key, value in key_values.items():
            self._logger.info(f"Step {step} - {key}: {value}")

    def close(self):
        pass

In [11]:
logger = get_file_logger(RUN_NAME, LOG_LEVEL)
ppo_logger = PpoLogger(
    folder=None,
    output_formats=[PpoKvWriter(logger)]
)
base_env = MultiAttemptSuperMarioWorldEmulator(level = LEVEL,
                                               render_mode = "rgb_array",
                                               reward_model = TryThingsRewardModel(),
                                               attempts = ATTEMPTS,
                                               render_debug = True,
                                               render_grid = True,
                                               logger = logger)
env = SuperMarioDiscretizer(base_env)
encoder = TileEncoder()

2026-05-05 02:14:33 [INFO] Session log for run cnn_action_rewards with level [INFO] initialized at: cnn_action_rewards_20260505_021433.log
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
try:
    distinct_observations = []
    last_distinct_obs = None
    obs = env.reset()
    done = False
    while not done:
        env.render()
        obs, reward, terminated, truncated, info = env.step(SuperMarioCombo.get_combo_id(SuperMarioCombo.RIGHT))
        if last_distinct_obs is None or not np.array_equal(obs, last_distinct_obs):
            last_distinct_obs = obs
            distinct_observations.append(obs)
        done = terminated or truncated
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

In [13]:
encoder.train(distinct_observations, print_progress=True)

Epoch 5/5: 100%|██████████| 2/2 [00:00<00:00, 89.64it/s, loss=6.3]


In [16]:
class ApplyEncodingWrapper(gym.ObservationWrapper):
    def __init__(self, env: gym.Env, encoder: TileEncoder, device: Optional[str] = None):
        super().__init__(env)
        self._encoder = encoder
        self.device = device
        if self.device is not None:
            self._encoder
        self.observation_space = gym.spaces.Box(
            low=0,
            high=255,
            shape=(16, 14, 16),
            dtype=np.uint8
        )

    def observation(self, observation: np.ndarray) -> np.ndarray:
        features = self._encoder.embed(observation)
        features = features.permute(2, 0, 1).cpu().numpy()
        return (features * 255).astype(np.uint8)

In [17]:
encoded_env = ApplyEncodingWrapper(env, encoder, device)

In [18]:
try:
    ppo_env = DummyVecEnv([lambda: encoded_env])
    policy_kwargs = dict(
        features_extractor_class=CustomMarioCNN,
        features_extractor_kwargs=dict(features_dim=64),
    )
    model = PPO("CnnPolicy", ppo_env, policy_kwargs=policy_kwargs, verbose=1, learning_rate=0.0003, n_steps=10000, device=device)
    model.set_logger(ppo_logger)
    prime_policy_for_combo(model, SuperMarioCombo.RIGHT_RUN, ppo_env, logger, iterations=1)
    logger.info("Starting training...")
    model.learn(total_timesteps=60000)
    model.save("ppo_mario_test")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/preprocessing.py:22: UserWarning: Treating image space as channels-last, while second dimension was smallest of the three.
  warnings.warn("Treating image space as channels-last, while second dimension was smallest of the three.")
/usr/local/lib/python3.12/dist-packages/stable_baselines3/ppo/ppo.py:155: UserWarning: You have specified a mini-batch size of 64, but because the `RolloutBuffer` is of size `n_steps * n_envs = 10000`, after every 156 untruncated mini-batches, there will be a truncated mini-batch of size 16
We recommend using a `batch_size` that is a factor of `n_steps * n_envs`.
Info: (n_steps=10000 and n_envs=1)
  warnings.warn(
2026-05-05 02:17:53 [INFO] Priming policy to prefer 'RIGHT_RUN'


Using cuda device
Wrapping the env in a VecTransposeImage.


2026-05-05 02:17:53 [INFO] Priming complete
2026-05-05 02:17:53 [INFO] Starting training...
2026-05-05 02:18:45 [INFO] Step 10000 - time/iterations: 1
2026-05-05 02:18:45 [INFO] Step 10000 - time/fps: 193
2026-05-05 02:18:45 [INFO] Step 10000 - time/time_elapsed: 51
2026-05-05 02:18:45 [INFO] Step 10000 - time/total_timesteps: 10000
2026-05-05 02:19:47 [INFO] Step 20000 - train/learning_rate: 0.0003
2026-05-05 02:19:47 [INFO] Step 20000 - train/entropy_loss: -2.3859556899708547
2026-05-05 02:19:47 [INFO] Step 20000 - train/policy_gradient_loss: -0.005320236076167814
2026-05-05 02:19:47 [INFO] Step 20000 - train/value_loss: 0.987129940105081
2026-05-05 02:19:47 [INFO] Step 20000 - train/approx_kl: 0.014550108462572098
2026-05-05 02:19:47 [INFO] Step 20000 - train/clip_fraction: 0.1296377388535032
2026-05-05 02:19:47 [INFO] Step 20000 - train/loss: 1.4562448263168335
2026-05-05 02:19:47 [INFO] Step 20000 - train/explained_variance: 0.0006804466247558594
2026-05-05 02:19:47 [INFO] Step 20

In [19]:
from gymnasium.wrappers import RecordVideo

try:
    video_env = RecordVideo(encoded_env, video_folder="./", name_prefix=f"trial-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
2026-05-05 02:24:07 [INFO] Terminated: True
2026-05-05 02:24:07 [INFO] Truncated: False
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [20]:
try:
    ppo_env = DummyVecEnv([lambda: encoded_env])
    policy_kwargs = dict(
        features_extractor_class=CustomMarioCNN,
        features_extractor_kwargs=dict(features_dim=64),
    )
    model = PPO("CnnPolicy", ppo_env, policy_kwargs=policy_kwargs, verbose=1, learning_rate=0.0003, n_steps=32000, device=device)
    model.set_logger(ppo_logger)
    prime_policy_for_combo(model, SuperMarioCombo.RIGHT_RUN, ppo_env, logger, iterations=1)
    logger.info("Starting training...")
    model.learn(total_timesteps=500000)
    model.save("ppo_mario_test")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/preprocessing.py:22: UserWarning: Treating image space as channels-last, while second dimension was smallest of the three.
  warnings.warn("Treating image space as channels-last, while second dimension was smallest of the three.")
2026-05-05 02:24:08 [INFO] Priming policy to prefer 'RIGHT_RUN'
2026-05-05 02:24:08 [INFO] Priming complete
2026-05-05 02:24:08 [INFO] Starting training...


Using cuda device
Wrapping the env in a VecTransposeImage.


2026-05-05 02:26:54 [INFO] Step 32000 - train/learning_rate: 0.0003
2026-05-05 02:26:54 [INFO] Step 32000 - train/entropy_loss: -2.1605997828161643
2026-05-05 02:26:54 [INFO] Step 32000 - train/policy_gradient_loss: -0.004178956431888376
2026-05-05 02:26:54 [INFO] Step 32000 - train/value_loss: 17.627647488587982
2026-05-05 02:26:54 [INFO] Step 32000 - train/approx_kl: 0.012854761444032192
2026-05-05 02:26:54 [INFO] Step 32000 - train/clip_fraction: 0.050646894904458596
2026-05-05 02:26:54 [INFO] Step 32000 - train/loss: 3.303713321685791
2026-05-05 02:26:54 [INFO] Step 32000 - train/explained_variance: 0.5886703133583069
2026-05-05 02:26:54 [INFO] Step 32000 - train/n_updates: 60
2026-05-05 02:26:54 [INFO] Step 32000 - train/clip_range: 0.2
2026-05-05 02:26:54 [INFO] Step 32000 - time/iterations: 1
2026-05-05 02:26:54 [INFO] Step 32000 - time/fps: 192
2026-05-05 02:26:54 [INFO] Step 32000 - time/time_elapsed: 166
2026-05-05 02:26:54 [INFO] Step 32000 - time/total_timesteps: 32000
2026

In [21]:
from gymnasium.wrappers import RecordVideo

try:
    video_env = RecordVideo(encoded_env, video_folder="./", name_prefix=f"full-train-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
2026-05-05 03:17:36 [INFO] Terminated: True
2026-05-05 03:17:36 [INFO] Truncated: False
